# Milestone 6: Research Contribution & Advanced Analytics
**Assigned to:** 

> Run `load_dataset.ipynb` first to ensure `df` and `DATA_PATH` are available.


## Section 1: Novel Research Contribution

**Chosen Method**: XGBoost Regressor + SHAP Explainability for **Mortality Rate Prediction**

This contribution extends the Milestone 5 dashboard by adding a **Predictive Analytics** module that forecasts disease mortality and explains key driving factors.  
This is research-level because it combines predictive modeling with model interpretability (XAI).

---

In [7]:
# =====================================================
# MILESTONE 6 - MODEL TRAINING & RESEARCH CONTRIBUTION
# XGBoost + SHAP for Mortality Rate Prediction
# =====================================================

import pandas as pd
import numpy as np
import joblib
import os
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --------------------- Load Data ---------------------
PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'global_health_enriched.csv.gz'

print("Loading data...")
df = pd.read_csv(PROCESSED_PATH, compression='gzip')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

# ====================== FEATURE ENGINEERING ======================
feature_cols = [
    'Prevalence Rate (%)', 'Incidence Rate (%)', 'Healthcare Access (%)',
    'Doctors per 1000', 'Hospital Beds per 1000', 'Urbanization Rate (%)',
    'Per Capita Income (USD)', 'Education Index', 'Recovery Rate (%)',
    'Severity_Index', 'DALY_Intensity', 'Vaccine_Available_Flag',
    'Weighted_Time_Impact', 'Avg_Incidence_Disease'
]

cat_cols = ['Disease Category', 'Age Group', 'Gender', 'Treatment Type']

# Prepare features and target
X = df[feature_cols + cat_cols].copy()
y = df['Mortality Rate (%)']

# One-hot encoding for categorical variables
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(f"Features prepared → {X.shape[1]} columns")

# ====================== TRAIN-TEST SPLIT ======================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set: {X_test.shape[0]:,} samples")

# ====================== TRAIN XGBOOST MODEL ======================
import xgboost as xgb

print("\nTraining XGBoost model... (this may take a while)")

model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.08,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror',
    tree_method='hist'          # Faster training
)

model.fit(X_train, y_train)

# ====================== EVALUATION ======================
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n" + "="*60)
print("MODEL PERFORMANCE")
print("="*60)
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")
print("="*60)

# ====================== FEATURE IMPORTANCE ======================
import plotly.express as px

importances = model.feature_importances_
feature_names = X_train.columns

feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feat_imp.head(10))

# Plot
fig = px.bar(feat_imp.head(15), 
             x='Importance', 
             y='Feature', 
             orientation='h',
             title="Top 15 Feature Importance (XGBoost)")
fig.show()

# ====================== SAVE MODEL ======================
os.makedirs("models", exist_ok=True)
model_path = "models/mortality_predictor.pkl"

joblib.dump(model, model_path)
print(f"\nModel successfully saved at: {model_path}")

Loading data...
Loaded: 1,000,000 rows × 33 columns
Features prepared → 32 columns
Training set: 800,000 samples
Test set: 200,000 samples

Training XGBoost model... (this may take a while)

MODEL PERFORMANCE
RMSE : 0.1392
MAE  : 0.0502
R²   : 0.9976

Top 10 Most Important Features:
                       Feature  Importance
9               Severity_Index    0.727019
0          Prevalence Rate (%)    0.247145
26               Age Group_61+    0.001009
25             Age Group_36-60    0.000980
31  Treatment Type_vaccination    0.000964
14  Disease Category_bacterial    0.000956
12        Weighted_Time_Impact    0.000955
29      Treatment Type_surgery    0.000954
7              Education Index    0.000945
23      Disease Category_viral    0.000940



Model successfully saved at: models/mortality_predictor.pkl


## Section 2: Scalability and Data Handling

- Uses `pd.read_csv()` with compression for 1M rows
- Caching with `@st.cache_data`
- Model trained on sampled data then validated on full dataset
- Ready for incremental learning in future versions

---

In [ ]:
# Prepare Scalable Data Flow
# Example: define chunked loading, incremental refresh, or synthetic streaming behavior.


## Section 3: Validation and Performance Metrics

**Metrics**:
- RMSE, MAE, R² Score
- SHAP Summary & Force Plots for explainability

---

In [ ]:
# Evaluate Performance
# Example: compute model metrics, benchmark runtime, or compare baseline vs improved results.


## Section 4: Research Paper Draft Outline

**Title**: Predictive Modeling and Explainable AI for Global Disease Mortality Risk Assessment

- Abstract
- Introduction
- Literature Review
- Methodology (Data, Features, XGBoost + SHAP)
- Results & Discussion
- Conclusion & Future Work

---

## Section 5: Final System Integration and Demonstration Plan

**Integration**: Add a new "Prediction & Insights" tab in the Streamlit dashboard that loads the trained model and shows predictions + SHAP explanations.

**Demo Flow**:
1. Show interactive dashboard (Milestone 5)
2. Demonstrate prediction feature (Milestone 6)
3. Show SHAP explanations
4. Discuss research value